# Manual multi-goal ranking: does ordering transfer?

Select **GPU → Run all**. We manually position the ball in the same final **R34 region**. There is no planning, rollout or experience collection.

Train two fresh ranking adapters: a **single-goal control** and a **multi-goal model**. Compare both with raw MSE and the **frozen successful multi-goal distance adapter** from the previous experiment. Earlier notebooks/results remain preserved.

For a training reference G, choose a farther observation F and nearer observation N. Train **cost(F,G) > cost(N,G)**. Geometry supplies ordering only; MSE is not fitted to numerical Euclidean distance.

The same 15 shifted goals, including y=18,10,8, are reserved from all fitting roles. Goal 0, (50,42), is a seen-reference control. All fitting nodes are offered at once in R34; this is not another expanding curriculum.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
SOURCE_RUN = DRIVE / 'datasets/curriculum_cost_adapter/radial_distance_v1/20261106T101715235909Z'
DISTANCE_BASELINE = DRIVE / 'datasets/curriculum_cost_adapter/radial_multigoal_v1/20261106T184946814711Z'
assert (DISTANCE_BASELINE/'branches/multi/complete.json').is_file(), 'Set DISTANCE_BASELINE to your completed multi-goal distance run.'
assert (SOURCE_RUN/'run.json').is_file(), 'Set SOURCE_RUN to your existing radial_distance_v1 run folder.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1' / RUN_ID
CACHE = Path('/content/radial-multigoal-ranking-cache') / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-multigoal-ranking')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = 'a917f166301e40a027a0d904d4c8035afde16a1c'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-goal-sweep-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Manual geometry source:',SOURCE_RUN)
print('Frozen multi-goal distance comparator:',DISTANCE_BASELINE)
print('Reserved goals: same 15 near/far shifts as the previous sweep.')
print('Local cache:',CACHE)

## Pinned frozen encoder, renderer and prior distance checkpoint

The original radial source supplies manual geometry. The completed multi-goal distance run supplies the frozen comparison adapter. Its historical code fingerprint may differ; persistent inputs, source lineage and checkpoint provenance are verified. The old local cache is unnecessary.

In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
print('Ready. Encoder preparation and adapter fitting run in separate processes to release GPU memory.')


def command(operation,branch=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.multigoal_ranking_experiment',operation,
          '--output-dir',OUTPUT]
    if operation!='report':args+=['--cache-dir',CACHE,'--device','cuda']
    if operation=='prepare':args+=['--source-run',SOURCE_RUN,'--distance-baseline',DISTANCE_BASELINE,'--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT]
    if operation=='train':
        if branch not in ('single','multi'):raise ValueError('Choose single or multi branch.')
        args+=['--branch',branch]
    return args

## Construct manual reference / farther / nearer comparisons

Use the same original 116 candidate nodes. Remove reserved goal coordinate/image matches, deduplicate training images and exclude heldout grid queries from optimization.

Each valid triple is **(G,F,N)** with three distinct admitted nodes, both queries within 34 of G, and **F at least 0.5 simulation units farther from G than N**. These are synthetic comparisons, not steps from a trajectory.

Single-goal fitting uses the original G. Multi-goal fitting uses all admitted training nodes as possible G. Sampling is uniform over valid training references, then uniform over that reference's comparisons. Exact populations and sampled exposure are retained.

Fresh features are encoded once in float32 batches of eight. Preparation checks that changing the environment goal leaves fixed-position observations unchanged.

In [ ]:
started=time.monotonic()
shell(command('prepare'),cwd=REPO)
config=json.loads((OUTPUT/'config.json').read_text())
design=json.loads((OUTPUT/'data/design.json').read_text())
print('Fitting configuration:',config['fitting'])
print('Candidate / admitted / removed nodes:',len(design['candidate_ids']),len(design['train_ids']),len(design['removed_candidate_ids']))
print('Eligible ranking triples:',config['triplet_counts'])
print('Fixed shared training-only raw scale:',config['scale'])
print('Reserved goals:',[design['points'][i] for i in design['goal_ids'][1:]])
print(f'Preparation took {(time.monotonic()-started)/60:.2f} minutes.')

## Fit ranking only

`C(A,G) = MSE(adapter(z_A), adapter(z_G))`

`loss = mean(softplus(0.2 - (C(F,G) - C(N,G))/s))`

The fixed positive **s** is the median positive raw MSE on training pairs, common to both arms. It sets optimization units; it is not a distance calibration. There is **no Euclidean regression target and no anchor loss**. Gradients pass through G, F and N.

Both fresh models start at identity: seed 0, bottleneck 64, Adam lr 1e-4, gradient clip 1, **2,000 updates × 32 triples**. Each update computes 96 adapter endpoints, versus 64 in the earlier distance experiment. The comparison with that frozen model is therefore a comparison of fitting recipes.

The fixed training probes differ between arms and are never test observations. We log their loss, ranking accuracy and cost magnitudes separately from noisy minibatch loss. Ranking scores can change magnitude without representing physical distance.

In [ ]:
started=time.monotonic()
for branch in ('single','multi'):
    print('\nTRAINING',branch,flush=True)
    shell(command('train',branch),cwd=REPO)
    t=json.loads((OUTPUT/'branches'/branch/'training.json').read_text())
    print('Fixed training probe before / after:',t['probe_before'],t['probe_after'])
    print('Sampled comparisons / endpoint computations:',len(t['sampled_triplet_ids']),sum(t['endpoint_counts']))
    print('Active training references:',len(t['reference_ids']))
    print('Reference / farther / nearer exposures:',sum(t['reference_counts']),sum(t['far_counts']),sum(t['near_counts']))
print(f'Both ranking arms took {(time.monotonic()-started)/60:.2f} minutes.')

## Evaluate ordering on the same heldout queries

Every goal/model uses the same query positions: inside original R34, within 34 of all tested goals, and excluding fitting/reference coordinate or rendered-image matches.

- **Uphill:** cost increases on an adjacent grid move that physically approaches the goal. Lower is better; ties are separate.
- **Correct pair ranking:** among heldout query pairs separated by at least 0.5 in true goal-distance, the farther query has higher cost. Higher is better. Reversed and tied counts are saved.
- The numerical cost tolerance is `max(1e-7, 0.001 × common-region cost range)`, separately for each cost map, matching earlier directional diagnostics.

No distance MAE/RMSE or goal-specific calibration is applied to ranking costs. Full heatmaps include probes outside the shared main cohort. Goal comparisons are descriptive for this seed, not planning success rates.

In [ ]:
started=time.monotonic()
shell(command('evaluate'),cwd=REPO)
shell(command('report'),cwd=REPO)
r=json.loads((OUTPUT/'report.json').read_text())
from IPython.display import display,Image

def pct(v):return 'N/A' if v is None else f'{v:.1%}'

print('Source/comparator unchanged:',r['references_unchanged'])
print('Original cached/fresh goal consistency:',r['original_goal_consistency'])
print('Ranking costs have arbitrary units; no distance calibration.')
print('UPHILL: lower is better; same heldout query cohort for every row')
print(f"{'id':>3} {'goal':>19} {'status':>9} {'N':>5} {'raw':>8} {'single':>8} {'multi':>8} {'distance':>9} {'steps':>7} {'multi ties':>11} {'multi down':>11} {'multi tie%':>11}")
for j,g in enumerate(r['methods']['multi']):
    values=[r['methods'][b][j]['heldout_directional']['adapted_up_fraction'] for b in ('single','multi','distance')]
    print(f"{j:>3} {str(tuple(round(v,3) for v in g['goal'])):>19} {g['reference_status']:>9} {g['heldout_count']:>5} "
          f"{pct(g['heldout_directional']['raw_up_fraction']):>8} "+' '.join(f'{pct(v):>8}' for v in values)+
          f" {g['heldout_directional']['comparisons']:>7} {g['heldout_directional']['adapted_ties']:>11} {pct(g['heldout_directional']['adapted_decreasing_fraction']):>11} {pct(g['heldout_directional']['adapted_tie_fraction']):>11}")
print('\nCORRECT HELDOUT PAIR RANKING: higher is better; physical distance gap >=0.5')
print(f"{'id':>3} {'raw':>8} {'single':>8} {'multi':>8} {'distance':>9} {'pairs':>8} {'multi ties':>11}")
for j,g in enumerate(r['methods']['multi']):
    values=[r['methods'][b][j]['adapted_rank']['correct_fraction'] for b in ('single','multi','distance')]
    print(f"{j:>3} {pct(g['raw_rank']['correct_fraction']):>8} "+' '.join(f'{pct(v):>8}' for v in values)+
          f" {g['adapted_rank']['comparisons']:>8} {g['adapted_rank']['ties']:>11}")
for name in ('coverage','training_loss','goal_summary'):display(Image(filename=str(OUTPUT/f'{name}.png')))
print(f'Evaluation/report took {(time.monotonic()-started)/60:.2f} minutes.')

## Full-right heatmaps for all goals

Each panel has its **own unclipped scale**. Compare the location of low/high costs and their ordering, not colors or cost magnitudes across panels. Ranking MSE has arbitrary units; the last panel alone shows physical distance.

White points are offered fitting nodes. Dashed circle: original R34 region. Orange star: evaluated reference. Goal 0 is seen; all 15 shifted goals are withheld. Full maps include outside-common-region probes.

In [ ]:
for j,g in enumerate(r['methods']['multi']):
    print('GOAL',j,g['goal'],'| reference:',g['reference_status'])
    print('Raw / single-rank / multi-rank / distance self costs:',g['raw_self_cost'],
          r['methods']['single'][j]['adapted_self_cost'],g['adapted_self_cost'],r['methods']['distance'][j]['adapted_self_cost'])
    display(Image(filename=str(OUTPUT/f'goal_{j}_heatmaps.png')))

## Summarize withheld goals separately

The primary question is whether multi-goal ranking gives more consistently decreasing costs on unseen goals, especially y<22. Check both local uphill/ties and broader heldout ordering. Check the original-goal control separately for tradeoffs.

The distance-trained comparator uses the previous fitting recipe; the two fresh ranking arms isolate single versus multiple training references. No test score selects a checkpoint or stops fitting.

In [ ]:
rows=r['methods']['multi']
for label,indices in [('all withheld',list(range(1,len(rows)))),
                      ('nearby withheld',[j for j,g in enumerate(rows) if j and g['goal'][1]>=22]),
                      ('distant withheld (y<22)',[j for j,g in enumerate(rows) if j and g['goal'][1]<22])]:
    valid=[j for j in indices if rows[j]['heldout_count']>0]
    print('\n',label,':',len(valid),'goals')
    for method in ('single','multi','distance'):
        up=[r['methods'][method][j]['heldout_directional']['adapted_up_fraction'] for j in valid]
        accuracy=[r['methods'][method][j]['adapted_rank']['correct_fraction'] for j in valid]
        up=[v for v in up if v is not None];accuracy=[v for v in accuracy if v is not None]
        print(method,'mean per-goal uphill:',pct(sum(up)/len(up) if up else None),
              '| correct ranking:',pct(sum(accuracy)/len(accuracy) if accuracy else None))
    for comparator in ('single','distance'):
        chosen=[j for j in valid if rows[j]['heldout_directional']['adapted_up_fraction'] is not None]
        wins=sum(rows[j]['heldout_directional']['adapted_up_fraction']<r['methods'][comparator][j]['heldout_directional']['adapted_up_fraction'] for j in chosen)
        print('Multi-ranking reduces uphill versus',comparator,':',wins,'/',len(chosen))
print('Seen original-goal uphill single/multi/distance:',*[pct(r['methods'][b][0]['heldout_directional']['adapted_up_fraction']) for b in ('single','multi','distance')])
print('Results:',OUTPUT)

## Return the executed notebook

Download **File → Download → Download .ipynb** and attach it here. Drive retains the manual dataset, offered triples, training samples/exposure, checkpoints, reference tokens/images, provenance and all reports/PNG/PDF figures. Reports and missing plots work after local cache deletion.

New results are separate from the distance and earlier ranking runs. Completed steps validate inputs/hashes before reuse; partial steps require a fresh timestamp. Local synthetic checks verify the workflow; actual GPU results determine transfer.